# Econometric Analysis

Stationarity tests, OLS regression, ARIMA-residual analysis, and Granger causality testing the relationship between RBI communication sentiment and short-window G-Sec yield / USD/INR changes.

## Step 0 — Path setup

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import statsmodels.api as sm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\hp\Desktop\rbi-sentiment-market-forecast


## Step 1 — Load data

Loads the meeting-level merged dataset (61 rows) and, separately, the full daily G-Sec/FX series required for the level-series ADF test and the ARIMA step.

In [2]:
from paths import MERGED_CSV, GSEC_CSV, USDINR_CSV

merged = pd.read_csv(MERGED_CSV, parse_dates=["listed_date"])
print(f"Loaded {len(merged)} meetings")

gsec_raw = pd.read_csv(GSEC_CSV)
gsec_raw["Date"] = pd.to_datetime(gsec_raw["Date"], dayfirst=True, errors="coerce")
gsec_raw["Price"] = gsec_raw["Price"].astype(str).str.replace(",", "", regex=False).astype(float)
gsec_series = gsec_raw.dropna(subset=["Date"]).sort_values("Date").set_index("Date")["Price"]

fx_raw = pd.read_csv(USDINR_CSV, parse_dates=["date"]).sort_values("date")
fx_series = fx_raw.set_index("date")["usdinr"]

print(f"Daily G-Sec series: {len(gsec_series)} rows")
print(f"Daily FX series: {len(fx_series)} rows")

Loaded 61 meetings
Daily G-Sec series: 2439 rows
Daily FX series: 2592 rows


## Step 2 — Stationarity (ADF)

Raw yield/FX levels are expected to be non-stationary, motivating the use of changes (rather than levels) as the dependent variable throughout.

In [3]:
from statsmodels.tsa.stattools import adfuller

def run_adf(series, name):
    series = pd.Series(series).dropna()
    stat, pval, *_ = adfuller(series)
    verdict = "STATIONARY" if pval < 0.05 else "NON-STATIONARY (unit root)"
    print(f"{name:40s} ADF={stat:8.3f}  p={pval:.4f}  -> {verdict}")

run_adf(gsec_series, "G-Sec yield (daily level)")
run_adf(fx_series, "USD/INR (daily level)")
run_adf(merged["dyield_1d"], "1-day yield change")
run_adf(merged["dyield_3d"], "3-day yield change")
run_adf(merged["lexicon_score_x1000"], "Lexicon sentiment score")
run_adf(merged["finbert_score"], "FinBERT sentiment score")

G-Sec yield (daily level)                ADF=  -1.751  p=0.4048  -> NON-STATIONARY (unit root)
USD/INR (daily level)                    ADF=   0.586  p=0.9873  -> NON-STATIONARY (unit root)
1-day yield change                       ADF=  -6.085  p=0.0000  -> STATIONARY
3-day yield change                       ADF=  -6.075  p=0.0000  -> STATIONARY
Lexicon sentiment score                  ADF=  -2.828  p=0.0544  -> NON-STATIONARY (unit root)
FinBERT sentiment score                  ADF=  -3.321  p=0.0140  -> STATIONARY


## Step 3 — OLS regression

Three specifications — 1-day yield change, 3-day yield change, 1-day USD/INR change — each regressed on lexicon sentiment, controlling for the repo-rate-change control variable.

In [4]:
def run_ols(y_col, x_cols, df, label):
    data = df[x_cols + [y_col]].dropna()
    X = sm.add_constant(data[x_cols])
    Y = data[y_col]
    model = sm.OLS(Y, X).fit()
    print(f"\n{'='*70}\n{label}  (n={len(data)})\n{'='*70}")
    print(model.summary())
    return model

model_1d = run_ols("dyield_1d", ["lexicon_score_x1000", "repo_rate_change_bps"], merged,
                    "1-day G-Sec yield change ~ lexicon sentiment + repo rate change")


1-day G-Sec yield change ~ lexicon sentiment + repo rate change  (n=61)
                            OLS Regression Results                            
Dep. Variable:              dyield_1d   R-squared:                       0.069
Model:                            OLS   Adj. R-squared:                  0.037
Method:                 Least Squares   F-statistic:                     2.149
Date:                Wed, 19 Aug 2026   Prob (F-statistic):              0.126
Time:                        21:55:03   Log-Likelihood:                 50.268
No. Observations:                  61   AIC:                            -94.54
Df Residuals:                      58   BIC:                            -88.20
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------

In [5]:
model_3d = run_ols("dyield_3d", ["lexicon_score_x1000", "repo_rate_change_bps"], merged,
                    "3-day G-Sec yield change ~ lexicon sentiment + repo rate change")


3-day G-Sec yield change ~ lexicon sentiment + repo rate change  (n=61)
                            OLS Regression Results                            
Dep. Variable:              dyield_3d   R-squared:                       0.031
Model:                            OLS   Adj. R-squared:                 -0.003
Method:                 Least Squares   F-statistic:                    0.9167
Date:                Wed, 19 Aug 2026   Prob (F-statistic):              0.406
Time:                        21:55:09   Log-Likelihood:                 49.059
No. Observations:                  61   AIC:                            -92.12
Df Residuals:                      58   BIC:                            -85.79
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------

In [6]:
model_fx = run_ols("dusdinr_1d_pct", ["lexicon_score_x1000", "repo_rate_change_bps"], merged,
                    "1-day USD/INR %% change ~ lexicon sentiment + repo rate change")


1-day USD/INR %% change ~ lexicon sentiment + repo rate change  (n=61)
                            OLS Regression Results                            
Dep. Variable:         dusdinr_1d_pct   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                 -0.024
Method:                 Least Squares   F-statistic:                    0.3058
Date:                Wed, 19 Aug 2026   Prob (F-statistic):              0.738
Time:                        21:55:23   Log-Likelihood:                -40.765
No. Observations:                  61   AIC:                             87.53
Df Residuals:                      58   BIC:                             93.86
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------

## Step 4 — Diagnostics

Breusch-Pagan test for heteroskedasticity; Durbin-Watson statistic for residual autocorrelation.

In [7]:
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson

for name, model in [("1-day yield model", model_1d), ("3-day yield model", model_3d), ("1-day FX model", model_fx)]:
    bp_stat, bp_pval, _, _ = het_breuschpagan(model.resid, model.model.exog)
    dw = durbin_watson(model.resid)
    print(f"{name}:")
    print(f"  Breusch-Pagan p-value: {bp_pval:.4f}  ({'heteroskedasticity present' if bp_pval < 0.05 else 'no strong evidence of heteroskedasticity'})")
    print(f"  Durbin-Watson: {dw:.3f}  (near 2.0 = good, far from 2.0 = autocorrelation concern)")
    print()

1-day yield model:
  Breusch-Pagan p-value: 0.7785  (no strong evidence of heteroskedasticity)
  Durbin-Watson: 1.547  (near 2.0 = good, far from 2.0 = autocorrelation concern)

3-day yield model:
  Breusch-Pagan p-value: 0.4210  (no strong evidence of heteroskedasticity)
  Durbin-Watson: 1.569  (near 2.0 = good, far from 2.0 = autocorrelation concern)

1-day FX model:
  Breusch-Pagan p-value: 0.6057  (no strong evidence of heteroskedasticity)
  Durbin-Watson: 1.864  (near 2.0 = good, far from 2.0 = autocorrelation concern)



## Step 5 — ARIMA-residual check

Fits ARIMA(1,1,1) to the full daily yield series to remove its own autocorrelation structure, then tests whether sentiment explains the residual ("surprise") component at each meeting's t+1 trading day.

In [8]:
from statsmodels.tsa.arima.model import ARIMA

print("Fitting ARIMA(1,1,1) on the daily yield series...")
arima_fit = ARIMA(gsec_series, order=(1, 1, 1)).fit()
gsec_resid = arima_fit.resid
print("Done.")

def value_at_offset(event_date, series, offset):
    idx = series.index
    pos = idx.searchsorted(event_date, side="right") - 1
    p = pos + offset
    return series.iloc[p] if 0 <= p < len(series) else None

merged["gsec_arima_resid_tp1"] = merged["listed_date"].apply(lambda d: value_at_offset(d, gsec_resid, 1))

model_arima = run_ols("gsec_arima_resid_tp1", ["lexicon_score_x1000"], merged,
                       "ARIMA residual (t+1) ~ lexicon sentiment")

Fitting ARIMA(1,1,1) on the daily yield series...


C:\Users\hp\anaconda3_new\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\hp\anaconda3_new\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\hp\anaconda3_new\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


Done.

ARIMA residual (t+1) ~ lexicon sentiment  (n=61)
                             OLS Regression Results                             
Dep. Variable:     gsec_arima_resid_tp1   R-squared:                       0.012
Model:                              OLS   Adj. R-squared:                 -0.005
Method:                   Least Squares   F-statistic:                    0.7011
Date:                  Wed, 19 Aug 2026   Prob (F-statistic):              0.406
Time:                          21:55:43   Log-Likelihood:                 116.26
No. Observations:                    61   AIC:                            -228.5
Df Residuals:                        59   BIC:                            -224.3
Df Model:                             1                                         
Covariance Type:              nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------

## Step 6 — Granger causality

Tests whether sentiment at meeting t improves prediction of the yield change at meeting t, beyond the yield series' own lagged values (lags 1-3).

In [9]:
from statsmodels.tsa.stattools import grangercausalitytests

granger_data = merged[["dyield_1d", "lexicon_score_x1000"]].dropna()
print(f"Granger causality test (n={len(granger_data)}), lexicon sentiment -> 1-day yield change:\n")
gc_result = grangercausalitytests(granger_data[["dyield_1d", "lexicon_score_x1000"]], maxlag=3)

Granger causality test (n=61), lexicon sentiment -> 1-day yield change:


Granger Causality
number of lags (no zero) 1
ssr based F test:         F=0.1916  , p=0.6632  , df_denom=57, df_num=1
ssr based chi2 test:   chi2=0.2017  , p=0.6533  , df=1
likelihood ratio test: chi2=0.2014  , p=0.6536  , df=1
parameter F test:         F=0.1916  , p=0.6632  , df_denom=57, df_num=1

Granger Causality
number of lags (no zero) 2
ssr based F test:         F=0.9736  , p=0.3843  , df_denom=54, df_num=2
ssr based chi2 test:   chi2=2.1275  , p=0.3452  , df=2
likelihood ratio test: chi2=2.0900  , p=0.3517  , df=2
parameter F test:         F=0.9736  , p=0.3843  , df_denom=54, df_num=2

Granger Causality
number of lags (no zero) 3
ssr based F test:         F=0.0914  , p=0.9644  , df_denom=51, df_num=3
ssr based chi2 test:   chi2=0.3120  , p=0.9578  , df=3
likelihood ratio test: chi2=0.3112  , p=0.9579  , df=3
parameter F test:         F=0.0914  , p=0.9644  , df_denom=51, df_num=3


---

**Summary of results.** Sentiment is not statistically significant in any of the three OLS specifications (p = 0.25, 0.48, 0.70); the ARIMA-residual check and Granger causality test independently confirm the null result. Full interpretation and literature comparison are in the project report.